In [24]:
import torch
all_embeddings=torch.load('../data/go_all_embeddings.pt',weights_only=False)


In [26]:
all_embeddings.keys()


dict_keys(['terms', 'context_embeddings', 'go_embeddings'])

In [33]:
all_embeddings['go_embeddings'][[1,2]].shape


(2, 200)

In [3]:
from dataset_generating.basics import *

go=Ontology('../data/go.obo',with_rels=True)

In [15]:
alt_list=[]
obsolete_list=[]

for term in go.ont.keys():
    if go.ont[term]['namespace']=='molecular_function':
        for alt_id in go.ont[term]['alt_ids']:
            alt_list.append(alt_id)
        if go.ont[term]['is_obsolete']:
            obsolete_list.append(term)

In [19]:
f1=pd.read_pickle('../data/swissprot.pkl')

In [22]:
for x in f1.prop_annotations.values:
    if len(set(x)&set(alt_list))>0:
        print(x)

In [27]:
def load_data(swissprot_file):
    """
    Parses UniProtKB data file and loads list of proteins and their
    annotations to lists
    Args:
       swissprot_file (string): A path to the data file
    Returns:
       Tuple of 8 lists (proteins, accessions, sequences, string_ids,
       species, genes, interpros)
    """
    
    proteins = list()
    accessions = list()
    sequences = list()
    annotations = list()
    string_ids = list()
    species = list()
    genes = list()
    interpros = list()
    with gzip.open(swissprot_file, 'rt') as f:
        prot_id = ''
        prot_ac = ''
        seq = ''
        org = ''
        annots = list()
        strs = list()
        iprs = list()
        gene_id = ''
        for line in f:
            items = line.strip().split('   ')
            if items[0] == 'ID' and len(items) > 1:
                if prot_id != '':
                    proteins.append(prot_id)
                    accessions.append(prot_ac)
                    sequences.append(seq)
                    annotations.append(annots)
                    string_ids.append(strs)
                    species.append(org)
                    genes.append(gene_id)
                    interpros.append(iprs)
                prot_id = items[1]
                annots = list()
                strs = list()
                iprs = list()
                seq = ''
                gene_id = ''
            elif items[0] == 'AC' and len(items) > 1:
                prot_ac = items[1]
            elif items[0] == 'OX' and len(items) > 1:
                if items[1].startswith('NCBI_TaxID='):
                    org = items[1][11:]
                    end = org.find(' ')
                    org = org[:end]
                else:
                    org = ''
            elif items[0] == 'DR' and len(items) > 1:
                items = items[1].split('; ')
                if items[0] == 'GO':
                    go_id = items[1]
                    code = items[3].split(':')[0]
                    annots.append(go_id + '|' + code)
                elif items[0] == 'STRING':
                    str_id = items[1]
                    strs.append(str_id)
                elif items[0] == 'GeneID':
                    gene_id = items[1]
                elif items[0] == 'InterPro':
                    ipr_id = items[1]
                    iprs.append(ipr_id)
            elif items[0] == 'SQ':
                seq = next(f).strip().replace(' ', '')
                while True:
                    sq = next(f).strip().replace(' ', '')
                    if sq == '//':
                        break
                    else:
                        seq += sq

        proteins.append(prot_id)
        accessions.append(prot_ac)
        sequences.append(seq)
        annotations.append(annots)
        string_ids.append(strs)
        species.append(org)
        genes.append(gene_id)
        interpros.append(iprs)
    return proteins, accessions, sequences, annotations, string_ids, species, genes, interpros


In [30]:
import gzip
proteins, accessions, sequences, annotations, _, species, _, _ = load_data('../data/uniprot_sprot_2021_04.dat.gz')

In [31]:
df = pd.DataFrame({
    'proteins': proteins,
    'accessions': accessions,
    'sequences': sequences,
    'annotations': annotations,
    'species': species
})

In [40]:

annotations = []
for i, row in enumerate(df.itertuples()):
    annots = []
    for annot in row.annotations:
        go_id, code = annot.split('|')
        if is_exp_code(code):
            annots.append(go_id)
    # Ignore proteins without experimental annotations
    if len(annots) == 0:
        continue
    annotations.extend(annots)

In [13]:
import torch
embedding_data = torch.load('../data/esm_embeddings_3B_complete.pt')
protein_labels,_,protein_embeddings_all=embedding_data.values()


In [33]:
esm2 = [protein_embeddings_all[x] for x in range(protein_embeddings_all.shape[0])]
pd.DataFrame({'proteins':protein_labels,'esm2':esm2})

,proteins,esm2
0,10H_STRNX,"[0.010345675, -0.0007756354, 0.04307754, -0.00..."
1,11H_STRNX,"[-0.024879452, -0.014601011, 0.049158923, -0.0..."
2,11K_PAVHV,"[0.020165285, 0.017978532, 0.03389687, -0.0075..."
3,11S1_CARIL,"[-0.08667583, 0.076800644, 0.021268692, -0.047..."
4,11S1_MACIN,"[0.11019135, -0.05891384, 0.032138463, -0.0855..."
...,...,...
82281,C71BS_ARATH,"[-0.030726945, -0.013734694, 0.041923333, -0.0..."
82282,C71BT_ARATH,"[-0.022942994, -0.01404988, 0.05019736, -0.013..."
82283,C71BU_ARATH,"[-0.049239676, -0.012281431, 0.037467167, -0.0..."
82284,C71CU_SINHE,"[-0.020553, -0.004683075, 0.039969284, -0.0270..."


In [10]:
# for x in ['train','valid','test']:

pd.read_pickle(f'../../deepgo2/data/cc/test_data_new.pkl')

,index,proteins,accessions,genes,sequences,annotations,string_ids,orgs,interpros,exp_annotations,prop_annotations,cafa_target
76653,559908,YQ65_SCHPO,Q9Y7P9;,2538933,MTGTAKVSIPKAQPHKVPCVYLAGDLVFRPNAIELFDELKEICKDA...,"[GO:0005829|HDA, GO:0005634|HDA, GO:0070694|IB...",[4896.SPCC191.05c.1],284812,[IPR007710],"[GO:0005829, GO:0005634]","[GO:0005622, GO:0005634, GO:0005737, GO:004323...",True
47724,305387,PEA15_HUMAN,Q15121; B1AKZ3; O00511;,8682,MAEYGTLLQDLTNNITLEDLEQLKSACKEDIPSEKSEEITTGSAWF...,"[GO:0005829|IDA, GO:0005875|NAS, GO:0005654|ID...",[9606.ENSP00000357055],9606,"[IPR011029, IPR001875, IPR029546]","[GO:0005829, GO:0005654, GO:1902042, GO:0046325]","[GO:0034763, GO:2001234, GO:2001233, GO:001064...",True
19470,116348,EFN4_CAEEL,O44516;,176882,MKQFFEFLITTFLLLGLAAADEHIVYWNSTNSLFRNRQPTIEVRMG...,"[GO:0031225|IEA, GO:0030424|IDA, GO:0043025|ID...",[6239.F56A11.3],6239,"[IPR008972, IPR031328, IPR001799]","[GO:0030424, GO:0043025, GO:0042074, GO:000979...","[GO:0007399, GO:0007492, GO:0048869, GO:004429...",True
44307,287294,NUOG_ECOLI,P33602; P76489; P78184; P78185;,946762,MATIHVDGKEYEVNGADNLLEACLSLGLDIPYFCWHPALGSVGACR...,"[GO:0005737|IEA, GO:0030964|IDA, GO:0005886|ID...",[511145.b2283],83333,"[IPR036010, IPR001041, IPR009010, IPR006657, I...","[GO:0030964, GO:0005886, GO:0045272, GO:005153...","[GO:0071944, GO:0098803, GO:0045272, GO:007046...",True
7546,46900,BNI3L_MOUSE,Q9Z2F7; Q545J6;,12177,MSHLVEPPPPLHNNNNNCEEGEQPLPPPAGLNSSWVELPMNSSNGN...,"[GO:0005829|ISO, GO:0005783|ISO, GO:0016021|IE...",[10090.ENSMUSP00000022634],10090,[IPR010548],"[GO:0005739, GO:0042802, GO:0043065, GO:190314...","[GO:0072655, GO:0015031, GO:0044248, GO:001922...",True
...,...,...,...,...,...,...,...,...,...,...,...,...
60522,438940,SEC18_YEAST,P18759; D6VQ79; Q07067;,852372,MFKIPGFGKAAANHTPPDMTNMDTRTRHLKVSNCPNNSYALANVAA...,"[GO:0005829|HDA, GO:0005794|IDA, GO:0005795|IB...",[4932.YBR080C],559292,"[IPR003593, IPR041569, IPR009010, IPR003959, I...","[GO:0005829, GO:0005794, GO:0043332, GO:001688...","[GO:0015031, GO:0044248, GO:1901564, GO:000689...",True
68769,488554,TPO3_CANGA,Q6FQ03;,2889240,MVDQESLVSFSSETSQSINSDIDIESQQQPRQYIPSNEKDGNKERL...,"[GO:0016021|IEA, GO:0005886|IDA, GO:0022857|IE...",[5478.XP_447691.1],284593,"[IPR011701, IPR020846, IPR036259]","[GO:0005886, GO:0000296]","[GO:0071944, GO:0006811, GO:0071702, GO:011016...",False
56536,391121,RLBP1_HUMAN,P12271; B2R667;,6017,MSEGVGTFRMVPEEEQELRAQLEQLTTKDHGPVFGPCSQLPRHTLQ...,"[GO:0044297|IEA, GO:0005813|IDA, GO:0005829|ID...",[9606.ENSP00000268125],9606,"[IPR001251, IPR036865, IPR011074, IPR036273, I...","[GO:0005813, GO:0005829, GO:0005654, GO:000760...","[GO:0050953, GO:0050877, GO:0005856, GO:001563...",True
57727,405957,RR14_ORYSA,P0C465; P09096; Q6QY76; Q7G7B6;,3131436,MAKKSLIQRERKRQKLEQKYHLIRRSSKKKIRSKVYPLSLSEKTKM...,"[GO:0009507|IEA, GO:0009536|IC, GO:0005840|IEA...",[],4530,"[IPR001209, IPR023036, IPR018271]",[GO:0009536],"[GO:0005622, GO:0043231, GO:0005737, GO:011016...",False


In [1]:
import pandas as pd

pd.read_pickle('../data/swissprot.pkl')

,proteins,accessions,sequences,annotations,species,exp_annotations,prop_annotations
0,11K_PAVHV,P0DJZ0;,MQNNTTGMDTKSLKNCGQPKAVCTHCKHSPPCPQPGCVTKRPPVPP...,"[GO:0030430|IDA, GO:0039526|IEA]",648237,[GO:0030430],"[GO:0033643, GO:0018995, GO:0030430, GO:000557..."
1,11S1_CARIL,B5KVH4;,MAKPILLSIYLCLIIVALFNGCLAQSGGRQQHKFGQCQLNRLDALE...,"[GO:0019863|IEA, GO:0045735|IC, GO:0048316|IEP...",32201,"[GO:0045735, GO:0048316, GO:0010431]","[GO:0032501, GO:0048316, GO:0048731, GO:000000..."
2,11S2_SESIN,Q9XHP0;,MVAFKFLLALSLSLLVSAAIAQTREPRLTQGQQCRFQRISGAQPSL...,"[GO:0042735|NAS, GO:0045735|NAS, GO:0010431|IEP]",4182,[GO:0010431],"[GO:0032501, GO:0048316, GO:0048731, GO:000000..."
3,128UP_DROME,P32234; Q9V648;,MSTILEKISAIESEMARTQKNKATSAHLGLLKAKLAKLRRELISPK...,"[GO:0005737|IBA, GO:0005525|IDA, GO:0002181|IBA]",7227,[GO:0005525],"[GO:0032561, GO:0035639, GO:0003674, GO:004316..."
4,13KDA_SCYCA,P83011;,MIFTAXDRSAIEXV,"[GO:0005783|IEA, GO:0043231|IDA]",7830,[GO:0043231],"[GO:0005575, GO:0005622, GO:0043227, GO:004323..."
...,...,...,...,...,...,...,...
77642,ZYX_XENLA,A5H447;,MDPAAPATRMTSSFTINISTPSFYNPPKKFAPVVPPKPKINPFKAP...,"[GO:0005737|IEA, GO:0005925|ISS, GO:0001725|IS...",8355,"[GO:0008134, GO:0006357]","[GO:0009889, GO:0065007, GO:0044249, GO:005079..."
77643,ZZZ3_HUMAN,Q8IYH5; B7WPC6; Q6N004; Q6N070; Q8IYP0; Q8IYR1...,MAASRSTRVTRSTVGLNGLDESFCGRTLRNRSIAHPEEISSNSQVR...,"[GO:0005671|IBA, GO:0005730|IDA, GO:0005654|ID...",9606,"[GO:0005730, GO:0005654]","[GO:0005730, GO:0005622, GO:0043227, GO:004323..."
77644,ZZZ3_MOUSE,Q6KAQ7; Q3TMK6; Q3V189;,MVGTCHSMAASRSTRVTRSTVGLNGLDESFCGRTLRNRSIAHPEEI...,"[GO:0005671|IDA, GO:0005730|ISO, GO:0005654|IS...",10090,[GO:0005671],[]
77645,Z_LASSJ,O73557;,MGNKQAKAPESKDSPRASLIPDATHLGPQFCKSCWFENKGLVECNN...,"[GO:0044220|IEA, GO:0020002|IEA, GO:0016020|IE...",11622,[GO:0046761],"[GO:0046755, GO:0051701, GO:0019076, GO:004676..."


In [28]:
print(len(all_esm2))


77638


In [3]:
import pandas as pd


all_esm2=[]
for aspect in ['mf','cc','bp']:
    test = pd.read_pickle('../../deepgo2/data/{}/test_data.pkl'.format(aspect))
    test = test[['proteins','esm2']]
    train = pd.read_pickle('../../deepgo2/data/{}/train_data.pkl'.format(aspect))
    train = train[['proteins','esm2']]
    valid = pd.read_pickle('../../deepgo2/data/{}/valid_data.pkl'.format(aspect))
    valid = valid[['proteins','esm2']]
    all_aspect = pd.concat([train,test,valid],ignore_index=True)
    print(len(all_aspect))
    all_esm2.append(all_aspect)


all_esm2 = pd.concat(all_esm2, ignore_index=True)
all_esm2 = all_esm2.drop_duplicates(subset='proteins').reset_index(drop=True)

43279
59257
58729


In [2]:
import torch
data=torch.load('../data/esm_embeddings_3B_complete.pt')
protein_labels=data['labels']
esm2=data['embeddings']


In [45]:
len(set(protein_labels))


82286

In [4]:
esm_2025=pd.DataFrame({'proteins':protein_labels,'esm2':esm2.cpu().numpy().tolist()
})
final=pd.concat([esm_2025,all_esm2],ignore_index=True)
final=final.drop_duplicates(subset='proteins').reset_index(drop=True)

In [6]:
import numpy as np

In [10]:
proteins=final['proteins'].values
embeddings=final['esm2'].values

In [16]:
esm2

tensor([[ 0.0103, -0.0008,  0.0431,  ...,  0.0414, -0.1548, -0.0572],
        [-0.0249, -0.0146,  0.0492,  ...,  0.0634, -0.2143, -0.0289],
        [ 0.0202,  0.0180,  0.0339,  ..., -0.0163, -0.1306,  0.0607],
        ...,
        [-0.0492, -0.0123,  0.0375,  ...,  0.0535, -0.1735, -0.0290],
        [-0.0206, -0.0047,  0.0400,  ...,  0.0813, -0.2096, -0.0303],
        [-0.0163, -0.0024,  0.0442,  ...,  0.0540, -0.1833, -0.0396]],
       device='cuda:0')

In [18]:
final_embeddings=torch.stack([torch.tensor(x) for x in embeddings])

In [19]:
export = {
    "labels": proteins,              # list / array
    "sequence": [],                  # 空 list
    "embeddings": final_embeddings   # torch.Tensor
}

torch.save(export, "../data/esm_embeddings_3B_complete_2021&2025.pt")
